In [ ]:
##Data Collection

In [ ]:
##Create S3 bucket and copy your data(if not created already)

In [ ]:
!aws s3 mb s3://retail-demand-forecasting-datalake

In [ ]:
!aws s3 ls

In [ ]:
!aws s3 cp ../Data/train.csv \s3://retail-demand-forecasting-datalake/raw/corporacion_favorita/train/train.csv

!aws s3 cp ../Data/stores.csv \s3://retail-demand-forecasting-datalake/raw/corporacion_favorita/stores/stores.csv

!aws s3 cp ../Data/oil.csv \s3://retail-demand-forecasting-datalake/raw/corporacion_favorita/oil/oil.csv

!aws s3 cp ../Data/holidays_events.csv \s3://retail-demand-forecasting-datalake/raw/corporacion_favorita/holidays/holidays_events.csv

!aws s3 cp ../Data/transactions.csv \s3://retail-demand-forecasting-datalake/raw/corporacion_favorita/transactions/transactions.csv


In [ ]:
!aws s3 ls s3://retail-demand-forecasting-datalake/raw/corporacion_favorita/ --recursive

In [ ]:
##Athena Setup

In [ ]:
import boto3
import sagemaker
import pandas as pd
from pyathena import connect
session = boto3.session.Session()
region = session.region_name
sagemaker_session = sagemaker.Session()
bucket = sagemaker_session.default_bucket()
s3_staging_dir = f"s3://{bucket}/athena/staging/"

s3 = boto3.Session().client(service_name="s3", region_name=region)

In [ ]:
conn = connect(
    region_name=region,
    s3_staging_dir=s3_staging_dir
)
cursor = conn.cursor()

In [ ]:
bucket = "your-bucket-name"
database_name = "your-db-name"

#Select the paths accordingly below is just an instance
s3_train_path = f"s3://{bucket}/raw/train/"
s3_stores_path = f"s3://{bucket}/raw/stores/"
s3_oil_path = f"s3://{bucket}/raw/oil/"
s3_holidays_path = f"s3://{bucket}/raw/holidays/"
s3_transactions_path = f"s3://{bucket}/raw/transactions/"


In [ ]:
statement = "CREATE DATABASE IF NOT EXISTS {}".format(database_name)
print(statement)

In [ ]:
pd.read_sql(statement, conn)

In [ ]:
statement = "SHOW DATABASES"

df_show = pd.read_sql(statement, conn)
df_show.head(5)

In [ ]:
if database_name in df_show.values:
    ingest_create_athena_db_passed = True

In [ ]:
%store ingest_create_athena_db_passed

In [ ]:
##Set the parameters

#DB name
database_name = "your-db-name"

#Tables name
train_table = "train_sales"
stores_table = "stores"
oil_table = "oil"
holidays_table = "holidays_events"
transactions_table = "transactions"

#stagging area to store the output from Athena
s3_staging_dir = "your-stagging-path"

In [ ]:
##train_sales table
statement_train = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.train_sales (
    id INT,
    date STRING,
    store_nbr INT,
    family STRING,
    sales DOUBLE,
    onpromotion INT
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
    'separatorChar' = ',',
    'quoteChar'     = '\"'
)
LOCATION '{s3_train_path}'
TBLPROPERTIES (
    'skip.header.line.count'='1',
    'use.null.for.invalid.data'='true'
)
""".format(
    database_name,
    "train_sales",
    s3_train_path
)
print(statement_train)


In [ ]:
##stores table
statement_stores = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.stores (
    store_nbr INT,
    city STRING,
    state STRING,
    type STRING,
    cluster INT
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
    'separatorChar' = ',',
    'quoteChar'     = '\"'
)
LOCATION '{s3_stores_path}'
TBLPROPERTIES (
    'skip.header.line.count'='1',
    'use.null.for.invalid.data'='true'
)
""".format(
    database_name,
    "stores",
    s3_stores_path
)
print(statement_stores)


In [ ]:
##oil table
statement_oil = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.oil (
    date STRING,
    dcoilwtico DOUBLE
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
    'separatorChar' = ',',
    'quoteChar'     = '\"'
)
LOCATION '{s3_oil_path}'
TBLPROPERTIES (
    'skip.header.line.count'='1',
    'use.null.for.invalid.data'='true'
)
""".format(
    database_name,
    "oil",
    s3_oil_path
)
print(statement_oil)


In [ ]:
##holidays_events table
statement_holidays = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.holidays_events (
    date STRING,
    type STRING,
    locale STRING,
    locale_name STRING,
    description STRING,
    transferred BOOLEAN
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
    'separatorChar' = ',',
    'quoteChar'     = '\"'
)
LOCATION '{s3_holidays_path}'
TBLPROPERTIES (
    'skip.header.line.count'='1',
    'use.null.for.invalid.data'='true'
)
""".format(
    database_name,
    "holidays_events",
    s3_holidays_path
)
print(statement_holidays)


In [ ]:
drop_query = f"""
DROP TABLE IF EXISTS {database_name}.transactions
"""

pd.read_sql(drop_query, conn)

In [ ]:
create_query = f"""
CREATE EXTERNAL TABLE {database_name}.transactions (
    date STRING,
    store_nbr INT,
    transactions INT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION '{s3_transactions_path}'
TBLPROPERTIES (
    'skip.header.line.count'='1'
)
"""

pd.read_sql(create_query, conn)


In [ ]:
pd.read_sql(f"DROP TABLE IF EXISTS {database_name}.transactions", conn)

In [ ]:
#Show db created(retail_forecasting)

cursor.execute("SHOW DATABASES")
cursor.fetchall()

In [ ]:
#Show tables in db "retail_forecasting"
statement = "SHOW TABLES in {}".format(database_name)

df_show = pd.read_sql(statement, conn)
df_show.head(5)

In [ ]:
pd.read_sql(
    "SELECT COUNT(*) AS cnt FROM retail_forecasting.train_sales",
    conn
)

In [ ]:
statement = """
SELECT date, store_nbr, sales
FROM retail_forecasting.train_sales
ORDER BY date
LIMIT 10
"""

pd.read_sql(statement, conn)

In [ ]:
cursor.execute(statement_train)

In [ ]:
cursor.execute(f"DESCRIBE {database_name}.train_sales")
cursor.fetchall()


In [ ]:
pd.read_sql(
    f"SELECT id, date, store_nbr, sales FROM {database_name}.train_sales LIMIT 5",
    conn
)